In [3]:

from pyspark.sql.functions import (
    col,
    from_json,
    to_timestamp,
    hour,
    window,
    count,
    sum as spark_sum,
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    TimestampType,
)

transaction_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("card_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("merchant_category", StringType(), True),
    StructField("country", StringType(), True),
    StructField("event_time", TimestampType(), True),
])




bronze = spark.read.table("northmart_dev.bronze.fraud_transactions")

parsed = (
        bronze
        .withColumn(
            "transaction",
            from_json(
                col("value"),
                transaction_schema,
                {
                    "schemaLocationKey": "fraud_transaction_payload"
                }
            )
        )
        .select(
            "transaction.*",
            "partition",
            "offset",
            "kafka_timestamp"
        )
)

display(transaction_schema)
parsed.show(5)


StructType([StructField('transaction_id', StringType(), True), StructField('card_id', StringType(), True), StructField('amount', DoubleType(), True), StructField('merchant_category', StringType(), True), StructField('country', StringType(), True), StructField('event_time', TimestampType(), True)])

+--------------------+-----------+-------+-----------------+-------+--------------------+---------+------+--------------------+
|      transaction_id|    card_id| amount|merchant_category|country|          event_time|partition|offset|     kafka_timestamp|
+--------------------+-----------+-------+-----------------+-------+--------------------+---------+------+--------------------+
|de7549c3-91fc-47a...|CARD-007008|  91.36|          GROCERY|     FR|2026-08-21 14:33:...|        1|   264|2026-08-21 14:33:...|
|975a03a0-8dc8-445...|CARD-004002|1868.53|       RESTAURANT|     SG|2026-08-21 14:34:...|        1|   265|2026-08-21 14:34:...|
|9c503c17-9395-467...|CARD-004054|  75.87|       RESTAURANT|     NL|2026-08-21 14:34:...|        1|   266|2026-08-21 14:34:...|
|1b321ff0-6408-44c...|CARD-003136| 241.03|        ECOMMERCE|     DE|2026-08-21 14:34:...|        1|   267|2026-08-21 14:34:...|
|982cb546-5324-443...|CARD-014966| 161.92|       RESTAURANT|     BE|2026-08-21 14:34:...|        1|   26